In [0]:
# Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.decomposition import PCA

from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity

import warnings
warnings.filterwarnings("ignore")

# Set Default Values
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sns.set_style('whitegrid')

In [0]:
# Setting path for dataset from GitHub
data_url = 'https://raw.githubusercontent.com/swapnilsaurav/Dataset/master/mall_customer_preprocessing_dataset.csv'

# Reading the csv file from the URL
df = pd.read_csv(data_url)

# Store a raw copy of the dataset
df_raw = df.copy() 

## Task 1 - Data Understanding

In [0]:
# Checking the Dimensions of the data

print("Dataset shape:", df.shape)
print("The number of rows in the data is:", df.shape[0])
print("The number of columns in the data is:", df.shape[1])

In [0]:
# Checking different attributes in the dataset

print("Attribute names:\n")
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c}")

In [0]:
# Checking the data types of the attributes

print("Attribute data types:\n")
print(df.dtypes)

In [0]:
# Printing First 5 records of the dataset

print("First five records:\n")
print(df.head().to_string())

In [0]:
# Descriptive Statsitcs of Numerical Variables

print("Descriptive statistics — numeric attributes:\n")
print(df.describe().T.to_string())

In [0]:
# Descriptive Statsitcs of Categorical Variables

print("Descriptive statistics — categorical attributes:\n")
print(df.describe(include='object').T)

In [0]:
# Identifying Numerical and Categroical Variables

identifier_cols = ['CustomerID', 'CustomerName']
date_cols = ['JoinDate', 'LastPurchaseDate']
target_col = ['ChurnNextMonth']
numerical_cols = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] and c not in target_col]
categorical_cols = [c for c in df.columns
                     if c not in identifier_cols + date_cols + numerical_cols + target_col]

print("Identifier columns   :", identifier_cols)
print("Date columns         :", date_cols)
print("Numerical columns    :", numerical_cols)
print("Categorical columns  :", categorical_cols)
print("Target column        :", target_col)

### Dataset understanding — summary

The dataset contains **2,200 customer records** and **25 attributes** collected by an
e-commerce company. 

The different attributes are: 
1. demographic data (`Age`, `Gender`, `City`, `Country`),
2. financial/behavioural data (`AnnualIncome_INR`, `SpendingScore_1_100`, `AvgBasketValue_INR`,
`TotalPurchases`, `OnlinePurchases`, `StorePurchases`, `LoyaltyPoints`), 
3. engagement data (`VisitFrequency`, `MembershipTier`, `PreferredCategory`, `CouponUsed`, `SatisfactionRating_1_5`),
4. channel/technical metadata (`DeviceType`, `EmailProvider`, `PaymentMethod`), 
5. two date fields (`JoinDate`, `LastPurchaseDate`), 
6. two identifiers (`CustomerID`, `CustomerName`), and 
7. the target label `ChurnNextMonth` (1 = did **not** purchase again / churned, 0 = purchased again).

Although pandas infers most text columns as `object`, several of them are genuinely
**numeric-but-noisy** (e.g. `Age`, `AnnualIncome_INR`) and the two date columns are stored as
plain strings in **at least four different formats**, which is a data-type issue and needs to be addressed.

At first glance the data looks "real-world messy": inconsistent capitalisation,
stray whitespace, mixed units, out-of-range values and missing values are present in almost
every column — exactly what we would expect from real world data.


## Task 2 - Data Quality Assessment

In [0]:
# Checking Missing values

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing Percentage': missing_pct})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
missing_summary['Missing Percentage'] = missing_summary['Missing Percentage'].astype(str) + '%'
print("Columns with missing values:\n")
print(missing_summary)

In [0]:
# Checking Duplicate Rows
dup_rows = df.duplicated().sum()
print(f"Fully duplicated rows        : {dup_rows}")

# Checking Duplicate CustomerIDs
dup_ids = df['CustomerID'].duplicated().sum()
print(f"Duplicated CustomerID values : {dup_ids}")

In [0]:
# Inspect a few duplicate CustomerIDs -- are they true copies or ID collisions?
dup_id_rows = df[df['CustomerID'].duplicated(keep=False)].sort_values('CustomerID')
conflicting = dup_id_rows.groupby('CustomerID').apply(
    lambda g: g.nunique().drop('CustomerID', errors='ignore').max() > 1
)
print(f"Of the repeated {dup_ids} CustomerIDs, {conflicting.sum()} groups have CONFLICTING data")
print(f"(i.e. the same CustomerID was accidentally assigned to different customers).")

In [0]:
# Checking Inconsistent Values in Categorical Variables

print("Categorical columns :", categorical_cols)
print()

for c in categorical_cols:
    vals = df[c].dropna().unique()
    print(f"{c} ({len(vals)} distinct raw values):")
    print("   ", sorted(vals.tolist(), key=str))
    print()

In [0]:
# Define IQR Rule
def iqr_outlier_count(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum())

# Calculate Outliers
for c in numerical_cols:
    count = iqr_outlier_count(df[c])
    if count > 0:
        print(f"{c} ({count} outliers)")
        print()

In [0]:
# Checking Invalid Entries

issues = []

age_invalid = ((df.Age < 0) | (df.Age > 100)).sum()
issues.append(('Invalid value', 'Age', age_invalid, 'Negative age or age > 100 is impossible'))

income_invalid = (df.AnnualIncome_INR < 0).sum()
issues.append(('Invalid value', 'AnnualIncome_INR', income_invalid, 'Negative income is impossible'))

basket_invalid = (df.AvgBasketValue_INR < 0).sum()
issues.append(('Invalid value', 'AvgBasketValue_INR', basket_invalid, 'Negative purchase value is impossible'))

purchases_invalid = (df.TotalPurchases < 0).sum()
issues.append(('Invalid value', 'TotalPurchases', purchases_invalid, 'Negative purchase count is impossible'))

loyalty_invalid = (df.LoyaltyPoints < 0).sum()
issues.append(('Invalid value', 'LoyaltyPoints', loyalty_invalid, 'Negative loyalty points is impossible'))

rating_invalid = ((df.SatisfactionRating_1_5 < 1) | (df.SatisfactionRating_1_5 > 5)).sum()
issues.append(('Invalid value', 'SatisfactionRating_1_5', rating_invalid, 'Rating must be on a 1-5 scale'))

quality_table = pd.DataFrame(issues, columns=['Problem', 'Column', 'Count', 'Impact/Reason'])

print("Invalid values:\n")
print(quality_table.to_string(index=False))

In [0]:
# Checking Data Type issues

print("JoinDate sample raw values       :", df['JoinDate'].sample(6, random_state=1).tolist())
print("LastPurchaseDate sample raw values:", df['LastPurchaseDate'].sample(6, random_state=1).tolist())
print()

print("-> Both date columns are stored as free-text strings mixing at least 4 formats:")
print("   DD-Mon-YYYY, MM/DD/YYYY, YYYY-MM-DD, MM-DD-YYYY")
print()
print()

print("CouponUsed raw values:", sorted(df['CouponUsed'].dropna().unique().tolist(), key=str))
print()
print("-> CouponUsed mixes numeric flags (0/1) and text flags (Y/N/Yes/No/yes/no) for the same boolean concept.")

### Summary Table of Data Quality Issues

| Problem | Column(s) | Count | Impact |
|---|---|---|---|
| Missing values | 20 of 25 columns (worst: `Country` 362, `VisitFrequency` 259, `CouponUsed` 265, `MembershipTier` 243, `Gender` 230, `DeviceType` 254, `EmailProvider` 228, `PaymentMethod` 220) | up to 16% per column | Reduces usable rows/features; biases summary statistics if not handled |
| Duplicate records | Full-row duplicates | 64 rows | Inflates counts, double-weights those customers during training |
| Duplicate primary key | `CustomerID` reused for different customers | 113 repeated IDs (many with genuinely conflicting attributes) | Breaks the 1-row-per-customer assumption; corrupts any join/merge on `CustomerID` |
| Inconsistent categories | `Gender`, `MembershipTier`, `VisitFrequency`, `PreferredCategory`, `City`, `Country`, `EmailProvider`, `DeviceType`, `PaymentMethod`, `CouponUsed` | 6-30 raw variants per column (case, spacing, spelling, abbreviations) | Same real-world category is fragmented into many levels, weakening group-by analysis and one-hot encoding |
| Invalid / impossible values | `Age` (<0 or >100), `AnnualIncome_INR` (<0), `AvgBasketValue_INR` (<0), `TotalPurchases` (<0), `LoyaltyPoints` (<0), `SatisfactionRating_1_5` (outside 1-5) | dozens of rows per column | Physically impossible values distort means, correlations and model training |
| Extreme statistical outliers (IQR rule) | `AnnualIncome_INR`, `AvgBasketValue_INR`, `TotalPurchases`, `LoyaltyPoints` | tens to hundreds of rows | Long right tails (e.g. income up to ₹10 crore, basket value up to ₹999,999) will dominate distance-based and scale-sensitive models |
| Data type issues | `JoinDate`, `LastPurchaseDate` stored as text in 4 mixed formats; `CouponUsed` mixes numeric and text boolean flags | all rows | Cannot compute tenure/recency or filter by boolean flag without parsing/casting first |


## Task 3 -  Data Cleaning

For every step below we show the **code**, the state **before**, the state **after**, and
the **reason** for the choice made.

### Step 3.1 — Remove exact duplicate rows

In [0]:
# Removing Duplicate Rows

before_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_rows = len(df)
print(f"BEFORE: {before_rows} rows")
print(f"AFTER : {after_rows} rows  ({before_rows - after_rows} exact duplicates removed)")
print("REASON: Fully identical rows are almost certainly duplicate data-entry / export artefacts and would double-count the same customer if left in.")

### Step 3.2 — Resolve `CustomerID` collisions (same ID, different customer)

In [0]:
# Remove Duplicate Customer IDs

dup_id_rows = df[df['CustomerID'].duplicated(keep=False)]
conflicting_ids = dup_id_rows.groupby('CustomerID').apply(
    lambda g: g.nunique().drop('CustomerID', errors='ignore').max() > 1
)
n_conflicting_groups = conflicting_ids.sum()
print(f"BEFORE: {df['CustomerID'].duplicated().sum()} duplicated CustomerID values, "
      f"{n_conflicting_groups} of which are genuine collisions (different customers).")

# Give every row a guaranteed-unique surrogate key, keep the original as CustomerID_Original
df['CustomerID_Original'] = df['CustomerID']
df['CustomerID'] = ['CUST' + str(100000 + i) for i in range(len(df))]

print(f"AFTER : {df['CustomerID'].duplicated().sum()} duplicated CustomerID values (by construction, 0).")
print("REASON: CustomerID should be a unique primary key. Where the same ID was reused for clearly different customers (different name/age/etc.), keeping it as-is would silently merge two different people. We regenerate a guaranteed-unique key and preserve the original value in CustomerID_Original for traceability.")


### Step 3.3 — Standardise whitespace in all text columns

In [0]:
# Removing Whitespace

text_cols = ['Gender', 'IncomeCurrency', 'MembershipTier', 'VisitFrequency','PreferredCategory', 'City', 'Country', 'EmailProvider','DeviceType', 'PaymentMethod', 'CouponUsed']

print("BEFORE:\n")
for c in text_cols:
    before = df.loc[df[c].notna(), c].unique()[:8]
    print(f"{c} raw values :", list(before))
print()

for c in text_cols:
    df[c] = df[c].astype('string').str.strip()

print()
print("AFTER:\n")
for c in text_cols:
    after = df.loc[df[c].notna(), c].unique()[:8]
    print(f"{c} raw values :", list(after))
print()

print("REASON: Leading/trailing spaces (e.g. ' GOLD ') make different values look the same, and these identical categories register as different strings, fragmenting group-bys and one-hot encoding.")

### Step 3.4 — Standardise `Gender`

In [0]:
# Standardising Gender Column Values

gender_map = {
    'female': 'Female',
    'f': 'Female', 
    'male': 'Male', 
    'm': 'Male',
    'other': 'Other'
}

before_vals = df['Gender'].value_counts(dropna=False)
df['Gender'] = df['Gender'].str.lower().map(gender_map).fillna(df['Gender'])
after_vals = df['Gender'].value_counts(dropna=False)

print("BEFORE:\n", before_vals)
print()
print("AFTER:\n", after_vals)
print()
print("REASON: 'Female','FEMALE','female','F' etc. all describe the same category; mapping to Title Case with 3 levels (Male/Female/Other) removes redundant categories.")

### Step 3.5 — Standardise `MembershipTier`

In [0]:
# Standardising MembershipTier Column Values

tier_map = {
    'basic': 'Basic', 
    'silver': 'Silver', 
    'silvr': 'Silver',
    'gold': 'Gold', 
    'platinum': 'Platinum'
}
before_vals = df['MembershipTier'].value_counts(dropna=False)
df['MembershipTier'] = df['MembershipTier'].str.lower().map(tier_map).fillna(df['MembershipTier'])
after_vals = df['MembershipTier'].value_counts(dropna=False)

print("BEFORE:\n", before_vals)
print()
print("AFTER:\n", after_vals)
print()
print("REASON: 'Silvr' is a typo of 'Silver'; case differences ('GOLD','gold') are the same tier.")

### Step 3.6 — Standardise `VisitFrequency`

In [0]:
# Standardising VisitFrequency Column Values

vf_map = {
    'daily': 'Daily', 
    'weekly': 'Weekly', 
    'monthly': 'Monthly', 
    'rarely': 'Rarely'
    }

before_vals = df['VisitFrequency'].value_counts(dropna=False)
df['VisitFrequency'] = df['VisitFrequency'].str.lower().map(vf_map).fillna(df['VisitFrequency'])
after_vals = df['VisitFrequency'].value_counts(dropna=False)

print("BEFORE:\n", before_vals)
print()
print("AFTER:\n", after_vals)
print()
print("REASON: Case-only variants of the same ordinal frequency category.")

### Step 3.7 — Standardise `PreferredCategory`

In [0]:
# Standardising PreferredCategory Column Values

cat_map = {
    'GAMING': 'Gaming',
    'gaming': 'Gaming',
    'sports': 'Sports', 
    'SPORTS': 'Sports', 
    'FASHION': 'Fashion',
    'fashion': 'Fashion',
    'HOME DECOR': 'Home Decor',
    'home decor': 'Home Decor',
    'beauty': 'Beauty', 
    'BEAUTY': 'Beauty', 
    'KIDS': 'Kids', 
    'kids': 'Kids', 
    'FOOD COURT': 'Food Court',
    'food court': 'Food Court',
    'fashion': 'Fashion',
    'books': 'Books',
    'BOOKS': 'Books',
    'electronics': 'Electronics',
    'Electronic': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'Eletronics': 'Electronics',
    'Groceries': 'Grocery',
    'grocery': 'Grocery',
    'GROCERY': 'Grocery',
    'Grocerry': 'Grocery'
}

before_vals = df['PreferredCategory'].value_counts(dropna=False)
df['PreferredCategory'] = df['PreferredCategory'].map(cat_map).fillna(df['PreferredCategory'])
after_vals = df['PreferredCategory'].value_counts(dropna=False)

print("BEFORE:\n", before_vals)
print()
print("AFTER:\n", after_vals)
print()
print("REASON: Title-casing merges case variants; 'Groceries' is standardised to 'Grocery' so that the singular/plural spelling of the same category is not double counted.")


### Step 3.8 — Standardise `City`

In [0]:
# Standardising City Column Values

city_map = {
    'bengaluru': 'Bengaluru', 
    'bangalore': 'Bengaluru',
    'mumbai': 'Mumbai', 
    'mumbay': 'Mumbai',
    'kolkata': 'Kolkata', 
    'calcutta': 'Kolkata',
    'hyderabad': 'Hyderabad', 
    'hyd': 'Hyderabad',
    'delhi': 'Delhi', 
    'new delhi': 'Delhi',
    'pune': 'Pune', 
    'jaipur': 'Jaipur', 
    'lucknow': 'Lucknow',
    'ahmedabad': 'Ahmedabad',
    'chennai': 'Chennai', 
    'chenai': 'Chennai'
}

before_vals = df['City'].value_counts(dropna=False)
df['City'] = df['City'].str.lower().map(city_map).fillna(df['City'])
after_vals = df['City'].value_counts(dropna=False)

print("BEFORE (top 12):\n", before_vals.head(12))
print()
print("AFTER (top 12):\n", after_vals.head(12))
print()
print("REASON: 'Mumbay' is a typo of 'Mumbai'; 'Bangalore'/'Calcutta' are older English names for 'Bengaluru'/'Kolkata'; 'Hyd' is an abbreviation for 'Hyderabad'.")


### Step 3.9 — Standardise `Country`

In [0]:
# Standardising Country Column Values

country_map = {
    'india': 'India', 
    'bharat': 'India', 
    'in': 'India'}

before_vals = df['Country'].value_counts(dropna=False)
df['Country'] = df['Country'].str.lower().map(country_map).fillna(df['Country'])
after_vals = df['Country'].value_counts(dropna=False)

print("Country BEFORE:\n", before_vals)
print()
print("Country AFTER:\n", after_vals)
print()
print("REASON: 'Bharat'/'IN'/'india' are all India")

### Step 3.10 — Standardise `IncomeCurrency`

In [0]:
# Standardising IncomeCurrency Column Values

# IncomeCurrency: 'Rs', 'INR', '₹' all denote the same currency as the AnnualIncome_INR column name implies. 
# 'USD' rows are a labelling error and are also standardised to INR.

curr_before = df['IncomeCurrency'].value_counts(dropna=False)
df['IncomeCurrency'] = df['IncomeCurrency'].str.upper().replace({'RS': 'INR', '₹': 'INR'})
df['IncomeCurrency'] = df['IncomeCurrency'].replace({'USD': 'INR'})
df['IncomeCurrency'] = df['IncomeCurrency'].fillna('INR')
curr_after = df['IncomeCurrency'].value_counts(dropna=False)

print("IncomeCurrency BEFORE:\n", curr_before)
print()
print("IncomeCurrency AFTER:\n", curr_after)
print()
print("REASON: 'Rs' and '₹' are symbols for INR; the 'USD' label was a data-entry error since the magnitude of those incomes is identical to the INR-labelled rows.")

### Step 3.11 — Standardise `EmailProvider`, `DeviceType`, `PaymentMethod`

In [0]:
# Standardising EmailProvider, DeviceType, PaymentMethod Column Values

for c, mapping in [
    ('EmailProvider', {}),
    ('DeviceType', {'android': 'Mobile', 'ios': 'Mobile', 'mobile': 'Mobile',
                     'web': 'Desktop', 'desktop': 'Desktop'}),
    ('PaymentMethod', {'upi': 'UPI', 'credit card': 'Credit Card', 'card': 'Credit Card',
                        'debit card': 'Debit Card', 'wallet': 'Wallet', 'cash': 'Cash'}),
]:
    before_vals = df[c].value_counts(dropna=False)
    if mapping:
        df[c] = df[c].str.lower().map(mapping).fillna(df[c].str.title())
    else:
        df[c] = df[c].str.lower()
    after_vals = df[c].value_counts(dropna=False)
    print(f"--- {c} ---")
    print("BEFORE:\n", before_vals)
    print()
    print("AFTER:\n", after_vals)
    print()

print("EmailProvider REASON: EmailProvider is lower-cased for consistency.")
print("DeviceType REASON: DeviceType is collapsed to the two channels that matter for the business (Mobile vs Desktop).") 
print("PaymentMethod REASON: PaymentMethod variants ('upi','UPI') and synonyms ('Card' -> 'Credit Card') are unified.")

### Step 3.12 — Standardise `CouponUsed`
Convert into a clean boolean (0/1)

In [0]:
# Standardising CouponUsed Column Values

coupon_map = {
    '1': 1, 
    '0': 0, 
    'y': 1, 
    'yes': 1, 
    'n': 0, 
    'no': 0
    }

before_vals = df['CouponUsed'].value_counts(dropna=False)
df['CouponUsed'] = df['CouponUsed'].str.lower().map(coupon_map)
after_vals = df['CouponUsed'].value_counts(dropna=False)

print("BEFORE:\n", before_vals)
print()
print("AFTER:\n", after_vals)
print()
print("REASON: The same yes/no concept was recorded as '1/0', 'Y/N' and 'Yes/No/yes/no'; mapping everything to an integer 0/1 flag makes the field directly usable by machine-learning algorithms.")

### Step 3.13 — Standardise `JoinDate` and `LastPurchaseDate`
Parse mixed-format dates

In [0]:
# Standardising JoinDate and LastPurchaseDate Column Values

sample_before = df[['JoinDate', 'LastPurchaseDate']].sample(5, random_state=7)
print("BEFORE (raw strings):\n", sample_before)

df['JoinDate'] = pd.to_datetime(df['JoinDate'], format='mixed', dayfirst=False, errors='coerce')
df['LastPurchaseDate'] = pd.to_datetime(df['LastPurchaseDate'], format='mixed', dayfirst=False, errors='coerce')

print()
print("AFTER (parsed datetime dtype):\n", df.loc[sample_before.index, ['JoinDate', 'LastPurchaseDate']])
print()
print("dtypes ->", "JoinDate:", df['JoinDate'].dtype, "LastPurchaseDate:", df['LastPurchaseDate'].dtype)
print(f"Unparseable dates -> NaT: JoinDate={df['JoinDate'].isna().sum()}, "
      f"LastPurchaseDate={df['LastPurchaseDate'].isna().sum()}")
print("REASON: pandas' 'mixed' format inference lets each row use its own layout (DD-Mon-YYYY, MM/DD/YYYY, YYYY-MM-DD, MM-DD-YYYY) and converts everything to a proper datetime64 dtype, which is required to compute tenure / recency features.")


`JoinDate` should always exist for an enrolled customer, so the handful of rows where
parsing failed are treated as a genuine (rare) missing-value problem and imputed with the
median join date. `LastPurchaseDate`, by contrast, can legitimately be unknown (a customer
who joined but has not purchased yet), so those remaining gaps are left as `NaT` rather than
imputed — filling them would fabricate a purchase event that may never have happened.

In [0]:
# Imputing Missing Values in JoinDate Column

before_na = df['JoinDate'].isna().sum()
df['JoinDate'] = df['JoinDate'].fillna(df['JoinDate'].median())
after_na = df['JoinDate'].isna().sum()

print(f"JoinDate missing BEFORE: {before_na}   AFTER: {after_na}")
print()
print("REASON: every customer must have a join date; the small number of parse failures are filled with the median join date rather than dropped, to avoid losing whole rows.")

### Step 3.14 — Convert impossible values to missing (`NaN`), then impute

In [0]:
# Convert Impossible Values to NaN

before_stats = df[['Age', 'AnnualIncome_INR', 'AvgBasketValue_INR', 'TotalPurchases',
                    'LoyaltyPoints', 'SatisfactionRating_1_5']].describe().T[['min', 'max', 'mean']]
print("BEFORE (min / max / mean of raw values, invalid entries included):")
print(before_stats)

# Recompute TotalPurchases as the reconciled sum of Online + Store purchases
df['TotalPurchases'] = df['OnlinePurchases'] + df['StorePurchases']

# Turn physically impossible values into NaN so they can be imputed like any other missing value
df.loc[(df['Age'] < 0) | (df['Age'] > 100), 'Age'] = np.nan
df.loc[df['AnnualIncome_INR'] < 0, 'AnnualIncome_INR'] = np.nan
df.loc[df['AvgBasketValue_INR'] < 0, 'AvgBasketValue_INR'] = np.nan
df.loc[df['LoyaltyPoints'] < 0, 'LoyaltyPoints'] = np.nan
df.loc[(df['SatisfactionRating_1_5'] < 1) | (df['SatisfactionRating_1_5'] > 5), 'SatisfactionRating_1_5'] = np.nan

after_stats = df[['Age', 'AnnualIncome_INR', 'AvgBasketValue_INR', 'TotalPurchases',
                   'LoyaltyPoints', 'SatisfactionRating_1_5']].describe().T[['min', 'max', 'mean']]
print()
print("AFTER (impossible values converted to NaN; TotalPurchases reconciled):")
print(after_stats)
print()
print("REASON: A negative age/income/basket-value/loyalty-points or a rating outside 1-5 cannot occur in reality -- these are data-entry/sensor errors, not genuine extreme customers, so we null them out rather than silently keep or clip them. TotalPurchases is derived from the two channel counts so it is always consistent.")


In [0]:
# Imputing Missing Values

missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
print("Missing values BEFORE imputation:\n", missing_before)

# Numeric columns -> median (robust to the remaining skew/outliers)
for col in ['Age', 'AnnualIncome_INR', 'AvgBasketValue_INR', 'LoyaltyPoints', 'SatisfactionRating_1_5']:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns -> mode, except where a more specific business rule applies
for col in ['Gender', 'MembershipTier', 'VisitFrequency', 'City', 'EmailProvider',
            'DeviceType', 'PaymentMethod']:
    df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

# Country: the dataset is overwhelmingly India, so missing country is filled with the mode
df['Country'] = df['Country'].fillna('India')

# CouponUsed: a missing flag most plausibly means no coupon was recorded as used
df['CouponUsed'] = df['CouponUsed'].fillna(0).astype(int)

missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)
print()
print("Missing values AFTER imputation (target + unimputed dates remain by design):\n", missing_after)
print()

print("REASON: Median imputation for numeric fields avoids distortion from the residual right-skew;")
print("        Mode imputation is standard practice for nominal categories;")
print("        Country and CouponUsed use explicit business defaults (majority country / 'no coupon').")
print("        LastPurchaseDate is left as NaT where unknown -- guessing a purchase date would fabricate behavioural history that was never observed.")
print("        ChurnNextMonth is the prediction target: rows with a missing label carry no training signal, so -- rather than guessing a label -- we will drop them.")

### Step 3.15 — Drop rows with a missing target label

In [0]:
# Dropping rows with missing target variable

before_rows = len(df)
before_missing_target = df['ChurnNextMonth'].isna().sum()
df = df.dropna(subset=['ChurnNextMonth']).reset_index(drop=True)
df['ChurnNextMonth'] = df['ChurnNextMonth'].astype(int)
after_rows = len(df)
print(f"BEFORE: {before_rows} rows, {before_missing_target} with a missing ChurnNextMonth label")
print()
print(f"AFTER : {after_rows} rows (0 missing labels)")
print()
print("REASON: ChurnNextMonth is the target variable for the planned ML model. A row with no label provides no supervised-learning signal, and imputing a label would inject fabricated ground truth, so these rows are removed instead.")

### Step 3.16 — Outlier treatment (winsorization / capping) on residual extreme values

In [0]:
# Plotting AnnualIncome_INR, AvgBasketValue_INR and LoyaltyPoints Before removing Outlier

cols_to_cap = ['AnnualIncome_INR', 'AvgBasketValue_INR', 'LoyaltyPoints']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cols_to_cap):
    sns.boxplot(y=df[col], ax=ax, color='#3f7cac')
    ax.set_title(f'{col}\n(Before Treatment)')
plt.tight_layout()
plt.savefig('fig_outliers_after.png', dpi=110, bbox_inches='tight')
plt.show()

In [0]:
# Removing Outliers

# Defining the function to cap the outliers
def cap_iqr(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return series.clip(lower=lo, upper=hi), lo, hi

before_desc = df[cols_to_cap].describe().T[['min', 'max', 'mean', 'std']]

# Capping the outliers
for col in cols_to_cap:
    capped, lo, hi = cap_iqr(df[col])
    df[col] = capped

after_desc = df[cols_to_cap].describe().T[['min', 'max', 'mean', 'std']]

print("BEFORE capping:\n", before_desc)
print()
print("AFTER capping:\n", after_desc)
print()
print("REASON: Even after removing physically-impossible values, a small number of extreme-but-technically-possible values remain (e.g. a basket value of ~₹999,999).")
print("        Left in, they would dominate the dataset and scale-sensitive models.")
print("        Winsorizing at 1.5xIQR keeps every row (no data loss) while bounding their influence.")

In [0]:
# Plotting AnnualIncome_INR, AvgBasketValue_INR and LoyaltyPoints After removing Outlier

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cols_to_cap):
    sns.boxplot(y=df[col], ax=ax, color='#3f7cac')
    ax.set_title(f'{col}\n(Treatment Done)')
plt.tight_layout()
plt.savefig('fig_outliers_after.png', dpi=110, bbox_inches='tight')
plt.show()

### Step 3.17 — Final data-type conversions

In [0]:
# Removing the data types to standardise the data

before_dtypes = df.dtypes.copy()
df['Age'] = df['Age'].round().astype(int)
df['CustomerName'] = df['CustomerName'].astype('string').str.strip()
df['ChurnNextMonth'] = df['ChurnNextMonth'].astype(int)
df['CouponUsed'] = df['CouponUsed'].astype(int)
after_dtypes = df.dtypes

changed = pd.DataFrame({'before': before_dtypes, 'after': after_dtypes})
changed = changed[changed['before'].astype(str) != changed['after'].astype(str)]
print("Columns whose dtype changed:\n", changed)
print()
print("REASON: Age and the binary flags are conceptually integers, not floats;")
print("         CustomerName whitespace is stripped for consistency with the other text fields.")

### Step 3.18 — Data Cleaning Summary

In [0]:
# Data Cleaning Summary

print(f"Raw dataset      : {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Cleaned dataset  : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Rows removed     : {df_raw.shape[0] - df.shape[0]}  Rows removed")

## Task 4 - Data Transformation

Making `df_model` a copy of the cleaned data used from here on so that the original cleaned
`df` keeps human-readable values and can be exported.

In [0]:
# Making a copy of df to do further transformation and preserve the cleanded dataset for exporting

df_model = df.drop(columns=['CustomerID_Original']).copy()
print("Working copy for transformation/reduction/proximity steps:", df_model.shape)

### 4.1 — Min-Max Normalization (`SpendingScore_1_100` -> [0,1])

In [0]:
# Min-max Normalisation for SpendingScore_1_100 Column

before = df_model['SpendingScore_1_100'].describe()[['min', 'max', 'mean']]
scaler_mm = MinMaxScaler()
df_model['SpendingScore_MinMax'] = scaler_mm.fit_transform(df_model[['SpendingScore_1_100']])
after = df_model['SpendingScore_MinMax'].describe()[['min', 'max', 'mean']]

print("BEFORE (SpendingScore_1_100):\n", before)
print()
print("AFTER  (SpendingScore_MinMax):\n", after)
print()
print("REASON: Rescaling to a common [0,1] range is useful for algorithms sensitive to scale (e.g. KNN, neural nets) and keeps the score directly comparable across features.")

### 4.2 — Standardization (Z-score) on `AnnualIncome_INR` and `AvgBasketValue_INR`

In [0]:
# Standard Scaling for AnnualIncome_INR and AvgBasketValue_INR Columns

cols_std = ['AnnualIncome_INR', 'AvgBasketValue_INR']
before = df_model[cols_std].describe().T[['mean', 'std']]
scaler_std = StandardScaler()
df_model[[c + '_Zscore' for c in cols_std]] = scaler_std.fit_transform(df_model[cols_std])
after = df_model[[c + '_Zscore' for c in cols_std]].describe().T[['mean', 'std']]

print("BEFORE (mean/std):\n", before)
print()
print("AFTER  (mean/std):\n", after)
print()
print("REASON: Income and basket value are on very different scales (lakhs vs hundreds of rupees).")
print("        Standardizing both to mean 0 / std 1 puts them on equal footing for  models like logistic regression / SVM that assume comparable scales.")

### 4.3 — Robust Scaling on `LoyaltyPoints`

In [0]:
# Robust Scaling for LoyaltyPoints Column

before = df_model['LoyaltyPoints'].describe()[['min', 'max', 'mean', 'std']]
scaler_rb = RobustScaler()
df_model['LoyaltyPoints_Robust'] = scaler_rb.fit_transform(df_model[['LoyaltyPoints']])
after = df_model['LoyaltyPoints_Robust'].describe()[['min', 'max', 'mean', 'std']]

print("BEFORE:\n", before)
print()
print("AFTER:\n", after)
print()
print("REASON: RobustScaler centers on the median and scales by the IQR instead of mean/std, making it less sensitive to the residual outliers in LoyaltyPoints than Min-Max or Standardization would be.")

### 4.4 — Log Transformation on `AnnualIncome_INR`

In [0]:
# Log Transformation for AnnualIncome_INR Column

before_skew = df_model['AnnualIncome_INR'].skew()
df_model['AnnualIncome_Log'] = np.log1p(df_model['AnnualIncome_INR'])
after_skew = df_model['AnnualIncome_Log'].skew()

print(f"BEFORE: skewness = {before_skew:.2f}")
print()
print(f"AFTER : skewness = {after_skew:.2f}")
print()
print("REASON: Income distributions are typically right-skewed (a few very high earners).")
print("        log1p compresses the long right tail, making the distribution closer to normal and better behaved for linear models.")

In [0]:
# Plotting AnnualIncome_INR Column data before and after log transformation

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df_model['AnnualIncome_INR'], bins=40, ax=axes[0], color='#e28743')
axes[0].set_title('AnnualIncome_INR (before log)')
sns.histplot(df_model['AnnualIncome_Log'], bins=40, ax=axes[1], color='#3f7cac')
axes[1].set_title('AnnualIncome_INR (after log1p)')
plt.tight_layout()
plt.savefig('fig_log_transform.png', dpi=110, bbox_inches='tight')
plt.show()

### 4.5 — Label Encoding on `MembershipTier`

In [0]:
# Label Encoding for MembershipTier Column

tier_order = {'Basic': 0, 'Silver': 1, 'Gold': 2, 'Platinum': 3}
before_vals = df_model['MembershipTier'].unique()
df_model['MembershipTier_Encoded'] = df_model['MembershipTier'].map(tier_order)
after_vals = df_model['MembershipTier_Encoded'].unique()

print("BEFORE (categories):", sorted(before_vals.tolist()))
print()
print("Maping: ", tier_order)
print()
print("AFTER  (ordinal codes):", sorted(after_vals.tolist()))
print()
print("REASON: MembershipTier has a natural order (Basic < Silver < Gold < Platinum), so a single ordinal code preserves that ranking.")

### 4.5 - One-Hot Encoding on nominal columns

In [0]:
# One-Hot Encoding for Gender, PaymentMethod, DeviceType Columns

nominal_cols = ['Gender', 'PaymentMethod', 'DeviceType']
before_shape = df_model.shape
df_model = pd.get_dummies(df_model, columns=nominal_cols, prefix=nominal_cols)
after_shape = df_model.shape

print(f"BEFORE one-hot encoding: {before_shape[1]} columns")
print()
print(f"AFTER  one-hot encoding: {after_shape[1]} columns "
      f"(+{after_shape[1] - before_shape[1]} dummy columns for {nominal_cols})")
print()
print("REASON: Gender/PaymentMethod/DeviceType have no natural order, so one-hot encoding avoids imposing a false ranking on them.")

## Task 5 - Data Reduction

We apply two reduction techniques: **(A) correlation-based feature removal** and
**(B) PCA**, both on the numeric feature set built in Task 4.

### 5.1 — Correlation-based feature removal

In [0]:
# Calculating correlation matrix

numeric_feats = ['Age', 'AnnualIncome_INR', 'SpendingScore_1_100', 'AvgBasketValue_INR',
                   'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'LoyaltyPoints',
                   'SatisfactionRating_1_5', 'CouponUsed', 'MembershipTier_Encoded']

corr = df_model[numeric_feats].corr()
print(f"BEFORE: {len(numeric_feats)} numeric features")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation matrix of numeric features')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()

In [0]:
# Feature removal bases on correlation matrix

# Identify pairs with |correlation| > 0.85  and drop one of each pair.
# We also explicitly flag TotalPurchases: even though its pairwise correlation with each
# individual channel is only moderate (~0.76-0.78, because Online/Store purchases are not
# perfectly correlated with each other), TotalPurchases is *structurally* redundant -- it was
# built in Task 3 as the exact sum OnlinePurchases + StorePurchases, so it carries zero
# information beyond what those two columns already provide.
threshold = 0.85
to_drop = set()
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if abs(corr.iloc[i, j]) > threshold:
            print(f"High statistical correlation: {cols[i]} <-> {cols[j]}  (r = {corr.iloc[i, j]:.2f})")
            to_drop.add(cols[j])

to_drop.add('TotalPurchases')  # structural redundancy, not purely statistical

reduced_feats = [c for c in numeric_feats if c not in to_drop]
print()
print("Columns dropped:", sorted(to_drop))
print(f"AFTER: {len(reduced_feats)} numeric features -> {reduced_feats}")
print()
print("REASON: TotalPurchases = OnlinePurchases + StorePurchases by construction (Task 3), so it")
print("        is linearly dependent on two other columns already in the feature set and adds no")
print("        new information, even though its pairwise Pearson correlation with each individual")
print(f"        channel ({corr.loc['TotalPurchases','OnlinePurchases']:.2f} and "
      f"{corr.loc['TotalPurchases','StorePurchases']:.2f}) does not itself cross the 0.85 threshold.")
print("        This shows why a purely statistical cutoff should be paired with domain knowledge about how features were derived.")

### 5.2 — Principal Component Analysis (PCA)

In [0]:
# Performing PCA on the features

pca_input = df_model[reduced_feats].copy()
pca_scaled = StandardScaler().fit_transform(pca_input)

pca_full = PCA().fit(pca_scaled)
explained = pca_full.explained_variance_ratio_
cum_explained = explained.cumsum()

print(f"BEFORE: {pca_input.shape[1]} features")
print("Explained variance ratio by component:")
for i, (ev, cev) in enumerate(zip(explained, cum_explained), 1):
    print(f"  PC{i}: {ev*100:5.1f}%   cumulative: {cev*100:5.1f}%")

In [0]:
# Calculating the number of principal components to retain

n_components = int(np.argmax(cum_explained >= 0.90) + 1)
pca_final = PCA(n_components=n_components)
pca_scores = pca_final.fit_transform(pca_scaled)
df_pca = pd.DataFrame(pca_scores, columns=[f'PC{i+1}' for i in range(n_components)])

print(f"AFTER: {n_components} principal components retain "
      f"{cum_explained[n_components-1]*100:.1f}% of the total variance "
      f"(reduced from {pca_input.shape[1]} to {n_components} dimensions).")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cum_explained)+1), cum_explained*100, marker='o', color='#3f7cac')
ax.axhline(90, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Number of components')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('PCA — cumulative explained variance')
plt.tight_layout()
plt.savefig('fig_pca_variance.png', dpi=110, bbox_inches='tight')
plt.show()

### - 5.3 Why these two techniques

- **Correlation-based removal** was chosen first because Task 2/3 already showed that
  `TotalPurchases` is mathematically derived from `OnlinePurchases` + `StorePurchases`; keeping
  all three would let the same signal be counted twice by any downstream model.
- **PCA** was chosen second because, even after removing the one redundant column, the
  remaining behavioural features (spending score, basket value, purchases, loyalty points,
  satisfaction, etc.) are moderately inter-correlated. PCA compresses them into a small number
  of uncorrelated components while retaining most of the variance — useful if the final model
  is distance-based (e.g. KNN, clustering) or if training speed / overfitting risk matters.



### 5.4 - Dataset size before vs after reduction

In [0]:
# Dataset Size before and After Reduction

print(f"Original numeric feature set      : {len(numeric_feats)} columns")
print(f"After correlation-based removal    : {len(reduced_feats)} columns")
print(f"After PCA (90% variance retained)  : {n_components} columns")

## Task 6 — Proximity Measures

We select 20 customers and compute pairwise **Euclidean distance** and **Cosine similarity**
on their standardized numeric profile (`reduced_feats`), then interpret which customers are
most similar / most different.

In [0]:
# Calculating Euclidean Distance and Cosine Similarity between customers

sample20 = df_model.sample(20, random_state=42).reset_index(drop=True)
sample_ids = sample20['CustomerID'].tolist()

X20 = StandardScaler().fit_transform(sample20[reduced_feats])

euclid = euclidean_distances(X20)
cosine = cosine_similarity(X20)

euclid_df = pd.DataFrame(euclid, index=sample_ids, columns=sample_ids)
cosine_df = pd.DataFrame(cosine, index=sample_ids, columns=sample_ids)

print("Selected 20 customers:", sample_ids)

In [0]:
# Visualizing the Euclidean Distance and Cosine Similarity between customers

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(euclid_df, cmap='viridis', ax=axes[0])
axes[0].set_title('Euclidean distance (lower = more similar)')
sns.heatmap(cosine_df, cmap='magma', ax=axes[1])
axes[1].set_title('Cosine similarity (higher = more similar)')
plt.tight_layout()
plt.savefig('fig_proximity_heatmaps.png', dpi=110, bbox_inches='tight')
plt.show()

In [0]:
# Finding the most similar pair by each measure (excluding the diagonal, i.e. a customer vs itself)
euclid_no_diag = euclid_df.where(~np.eye(len(euclid_df), dtype=bool))
min_pos = np.unravel_index(np.nanargmin(euclid_no_diag.values), euclid_no_diag.shape)
most_similar_euclid = (euclid_df.index[min_pos[0]], euclid_df.columns[min_pos[1]], euclid_df.values[min_pos])

cosine_no_diag = cosine_df.where(~np.eye(len(cosine_df), dtype=bool))
max_pos = np.unravel_index(np.nanargmax(cosine_no_diag.values), cosine_no_diag.shape)
most_similar_cosine = (cosine_df.index[max_pos[0]], cosine_df.columns[max_pos[1]], cosine_df.values[max_pos])

print(f"Most similar pair by Euclidean distance : {most_similar_euclid[0]} & {most_similar_euclid[1]} "
      f"(distance = {most_similar_euclid[2]:.3f})")
print(f"Most similar pair by Cosine similarity   : {most_similar_cosine[0]} & {most_similar_cosine[1]} "
      f"(similarity = {most_similar_cosine[2]:.3f})")
print()
cols_show = ['CustomerID'] + reduced_feats
print(sample20[sample20['CustomerID'].isin([most_similar_euclid[0], most_similar_euclid[1]])][cols_show].to_string(index=False))

### Interpretation
Euclidean distance measures the absolute difference in the raw
magnitude of each standardized feature, so two customers are flagged as "close" when *all*
their numeric attributes (age, income, spending score, basket value, purchases, loyalty
points, satisfaction, coupon use, membership tier) are simultaneously similar in scale.
Cosine similarity instead measures the similarity in *direction* of their feature vectors,
so it can rate two customers as similar even if one spends consistently more than the other,
as long as their relative pattern across features is the same (e.g. both are "high spender /
high loyalty / low satisfaction" shaped, just at different absolute levels). In this sample,
the pair flagged as most similar by Euclidean distance is a genuinely close match on every
raw attribute, which is the more intuitive notion of "look-alike customer" for a marketing
use case such as targeted offers or churn-risk look-alike modelling.

## Task 7 — Reflection Report

**Major preprocessing challenges.** The biggest challenge was the sheer *variety* of quality
problems packed into a fairly small (2,200-row) table: nearly every categorical column had
its own inconsistent-casing/whitespace/typo pattern (e.g. `'Silvr'`, `' GOLD '`, `'Mumbay'`,
`'Bharat'`), several numeric columns contained physically impossible values (negative ages,
ages above 100, negative income, ratings outside the stated 1-5 scale), and roughly half the
rows had a `TotalPurchases` value that contradicted its own component counts
(`OnlinePurchases` + `StorePurchases`). On top of that, the two date columns mixed at least
four different string formats, and 113 `CustomerID` values were reused across genuinely
different customers — a primary-key integrity problem that is easy to overlook if you assume
an ID column is trustworthy by default. Untangling which anomalies were genuine data noise 
(safe to fix or impute) versus which were legitimate rare events (safe to keep, only
scale/cap) required going column-by-column rather than applying one blanket rule.

**Which preprocessing step had the greatest impact?** Reconciling `TotalPurchases` with
`OnlinePurchases + StorePurchases` was arguably the single highest-impact step: it touched
~48% of rows and replaced an unreliable, occasionally absurd value (up to 9,999 purchases)
with a value that is guaranteed to be internally consistent and directly interpretable.
The second most effective was standardizing the categorical columns — collapsing `Gender`, 
`MembershipTier`,`City`, `PaymentMethod`, etc. down to their true number of categories 
dramatically improves the quality of any group-by analysis, and is essential before one-hot
encoding (otherwise the model would learn spurious "different" categories for `'Gold'` and `' GOLD '`).

**What problems remained?** A handful of `LastPurchaseDate` values are still missing — we
deliberately chose not to fabricate a purchase date, so any recency-based feature will need
to explicitly handle these as "unknown," not as "long ago." The dataset also still contains
some genuinely extreme (but not impossible) values after winsorization at the 1.5xIQR bound;
this reduces their influence but does not remove the underlying uncertainty about whether they
represent real high-value customers or further data-entry noise. Finally, standardizing
`CustomerID` collisions by regenerating a surrogate key resolves the symmetry problem but
cannot recover which of the conflicting original records is "correct".

**Recommended ML algorithm.** For predicting `ChurnNextMonth`, a **gradient-boosted tree
model** (e.g. XGBoost / LightGBM) is a strong first choice: tree ensembles handle the mix 
of numeric and categorical features gracefully, are naturally robust to the residual outliers 
and skew that remain even after cleaning, and typically give strong out-of-the-box performance 
on tabular churn-style problems without requiring extensive further scaling. 
A regularized **logistic regression** is a good complementary baseline for interpretability since 
coefficients are directly explainable, especially once the scaled/encoded features are used.

**Additional preprocessing before deployment.** Before shipping a production model we would
want to:

(1) re-validate the cleaning rules against a live, continuously-refreshed data feed,
since new unseen category spellings or date formats will keep appearing; 

(2) engineer explicit tenure (`today - JoinDate`) and recency (`today - LastPurchaseDate`) 
features now that the dates are true datetimes; 

(3) monitor for target/label leakage — e.g. confirm that `ChurnNextMonth` genuinely reflects 
*future* behaviour and was not partly derived from features already in the table;

(4) set up automated data-quality checks (schema validation, range checks, referential-integrity 
checks on `CustomerID`) so the issues found manually are caught automatically on every new data load; 

(5) periodically re-fit the scalers/encoders (or persist them) so that new data is transformed 
exactly the same way the training data was.


## Exporting the Cleaned Dataset

In [0]:
# Exporting the Cleaned Dataset in csv format

export_cols = [c for c in df.columns if c != 'CustomerID_Original']
output_path = "/Workspace/Users/kushagrakumar1205@gmail.com/bits_msc_ds_ai/Sem_1/Data_Pre_Processing/Assignment/mall_customer_cleaned.csv"
df[export_cols].to_csv(output_path, index=False)
print(f"Cleaned dataset exported: {df.shape[0]} rows x {len(export_cols)} columns")
print(f"Saved as {output_path}")